# 🐄 Cow Lameness Analysis v32 — Binary Classification
## Frozen VideoMAE + DLC Pose Features + Temporal Transformer

**Architecture:** DLC pose features + VideoMAE clip embeddings → Temporal Transformer → Binary (Healthy/Lame)

**Target:** Q1 Journal — ≥80% accuracy, 5-fold subject-level CV, ablation study, full evaluation suite

---
**ADR (Architecture Decision Record):**
| Decision | Choice | Rationale |
|----------|--------|-----------|
| Classification | Binary (0/1) | Folder-level labels only |
| Pose | DeepLabCut SuperAnimal | 1167 outputs ready |
| VideoMAE | Frozen backbone | Pre-compute embeddings once, train temporal head |
| Temporal | Transformer (4L, 8H) | Clip sequence reasoning |
| MIL | ❌ Removed | Redundant with Temporal Transformer |
| Validation | 5-fold subject-level CV | Q1 standard |


In [ ]:
# ============================================================
# SECTION 1: Environment, Imports & Configuration
# ============================================================
import os
import sys
import glob
import random
import warnings
import numpy as np
import pandas as pd
from pathlib import Path
from typing import Dict, List, Tuple, Optional
from collections import defaultdict

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, roc_curve, precision_recall_curve,
    confusion_matrix, classification_report, average_precision_score
)
from sklearn.model_selection import StratifiedGroupKFold
from scipy import stats
from scipy.signal import find_peaks

warnings.filterwarnings('ignore')
print("✅ Core imports successful")

# Install transformers if needed
try:
    from transformers import VideoMAEModel
    print("✅ Transformers already installed")
except ImportError:
    print("📦 Installing transformers...")
    os.system("pip install -q transformers accelerate")
    from transformers import VideoMAEModel
    print("✅ Transformers installed")

try:
    import cv2
    print("✅ OpenCV available")
except ImportError:
    os.system("pip install -q opencv-python-headless")
    import cv2
    print("✅ OpenCV installed")

print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")


In [ ]:
# ============================================================
# Configuration — Single source of truth
# ============================================================
CFG = {
    # Reproducibility
    "SEED": 42,

    # Data
    "VIDEO_DIR": "/content/drive/MyDrive/Inek Topallik Tespiti Parcalanmis Inek Videolari/cow_single_videos",
    "DLC_OUTPUT_DIR": "/content/drive/MyDrive/DeepLabCut/outputs",
    "RESULTS_DIR": "/content/drive/MyDrive/CowLameness_v32_results",

    # Clip extraction
    "NUM_CLIPS": 8,
    "CLIP_LENGTH": 16,
    "IMG_SIZE": 224,

    # Pose features
    "POSE_FRAMEWORK": "deeplabcut",
    "POSE_FEAT_DIM": 16,
    "MIN_CONFIDENCE": 0.3,

    # VideoMAE (Partial FT — blocks 10-11 trainable, hybrid pre-computation)
    "VIDEOMAE_MODEL": "MCG-NJU/videomae-base",
    "VIDEOMAE_DIM": 768,
    "PROJECTION_DIM": 256,
    "TRAINABLE_BLOCKS": [10, 11],

    # Temporal Transformer
    "HIDDEN_DIM": 256,
    "NUM_HEADS": 8,
    "NUM_LAYERS": 4,
    "DROPOUT": 0.3,

    # Training
    "BATCH_SIZE": 4,
    "EPOCHS": 40,
    "LR_VIDEOMAE": 1e-5,
    "LR_HEAD": 1e-4,
    "WEIGHT_DECAY": 1e-4,
    "PATIENCE": 7,
    "GRAD_CLIP": 1.0,
    "CV_FOLDS": 5,

    # Class labels
    "HEALTHY_LABEL": 0,
    "LAME_LABEL": 1,
}

# Deterministic everything
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    os.environ['PYTHONHASHSEED'] = str(seed)

set_seed(CFG["SEED"])
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"✅ Config loaded | Device: {DEVICE} | Seed: {CFG['SEED']}")


In [ ]:
# ============================================================
# Google Drive Mount
# ============================================================
try:
    from google.colab import drive
    drive.mount('/content/drive')
    print("✅ Google Drive mounted")
except Exception:
    print("⚠️ Not running in Colab — using local paths")

# Create results directory
os.makedirs(CFG["RESULTS_DIR"], exist_ok=True)
print(f"📁 Results will be saved to: {CFG['RESULTS_DIR']}")


---
## Section 2: Data Discovery & Subject ID Extraction

Discover all video files and their corresponding DLC outputs.
Extract `animal_id` from filenames for subject-level splitting.


In [ ]:
# ============================================================
# SECTION 2: Data Discovery & Subject ID Extraction
# ============================================================

def discover_data(cfg: dict) -> pd.DataFrame:
    """
    Discover videos and match with DLC pose outputs.

    Returns:
        DataFrame with columns: video_path, dlc_csv_path, label, animal_id
    """
    video_dir = Path(cfg["VIDEO_DIR"])
    dlc_dir = Path(cfg["DLC_OUTPUT_DIR"])

    records = []

    for folder, label in [("Saglikli", 0), ("Topal", 1)]:
        video_folder = video_dir / folder
        dlc_subfolder = dlc_dir / folder

        if not video_folder.exists():
            print(f"⚠️ Video folder not found: {video_folder}")
            continue

        videos = sorted(video_folder.glob("*.mp4"))
        print(f"📁 {folder}: {len(videos)} videos found")

        for vpath in videos:
            stem = vpath.stem

            # Extract animal_id
            parts = stem.replace("cow_", "").replace("cow", "")
            animal_id = parts.split("_")[0].split("DLC")[0]

            # Find matching DLC CSV (subfolder first, then root, then recursive)
            dlc_csv = None
            for search_dir in [dlc_subfolder, dlc_dir]:
                if search_dir.exists():
                    matches = list(search_dir.glob(f"{stem}DLC*.csv"))
                    if matches:
                        dlc_csv = str(matches[0])
                        break
            if dlc_csv is None and dlc_dir.exists():
                matches = list(dlc_dir.glob(f"**/{stem}DLC*.csv"))
                if matches:
                    dlc_csv = str(matches[0])

            records.append({
                "video_path": str(vpath),
                "dlc_csv_path": dlc_csv,
                "label": label,
                "label_name": folder,
                "animal_id": animal_id,
                "video_name": stem
            })

    df = pd.DataFrame(records)

    # Summary statistics
    print(f"\n{'='*50}")
    print(f"📊 Dataset Summary")
    print(f"{'='*50}")
    print(f"Total videos: {len(df)}")
    print(f"  Healthy (Sağlıklı): {(df['label']==0).sum()}")
    print(f"  Lame (Topal):       {(df['label']==1).sum()}")
    print(f"Unique animals: {df['animal_id'].nunique()}")
    print(f"DLC outputs available: {df['dlc_csv_path'].notna().sum()}/{len(df)}")
    print(f"{'='*50}")

    return df

data_df = discover_data(CFG)
data_df.head(10)


---
## Section 3: Pose Feature Extraction (DeepLabCut)

Extract 16 biomechanically meaningful features from DLC SuperAnimal outputs.

| Feature Group | Features | Clinical Meaning |
|---------------|----------|-----------------|
| Head bob | vertical displacement mean/std | Primary lameness indicator |
| Spine angle | mean/std, curvature variance | Postural compensation |
| Step timing | left/right duration mean/std | Gait regularity |
| Asymmetry | temporal ratio, step frequency | Core lameness signal |
| Hip sway | amplitude, range | Balance compensation |
| Stride | left/right CV, angle asymmetry | Movement consistency |


In [ ]:
# ============================================================
# SECTION 3: Pose Feature Extractor
# ============================================================

class PoseFeatureExtractor:
    """
    Extract biomechanical gait features from DLC SuperAnimal outputs.

    Produces a fixed-size feature vector per video/clip.
    Designed for future MMPose compatibility via framework parameter.
    """

    KEYPOINT_GROUPS = {
        "head": ["nose", "head", "forehead"],
        "withers": ["neck_base", "withers", "shoulder_center"],
        "spine": ["back_middle", "spine", "mid_back"],
        "tail_base": ["tail_base", "tailbase", "tail"],
        "left_front_hoof": ["front_left_paw", "left_front_paw", "lf_paw"],
        "right_front_hoof": ["front_right_paw", "right_front_paw", "rf_paw"],
        "left_hind_hoof": ["back_left_paw", "left_hind_paw", "lh_paw"],
        "right_hind_hoof": ["back_right_paw", "right_hind_paw", "rh_paw"],
        "left_front_knee": ["front_left_knee", "left_front_knee", "lf_knee"],
        "right_front_knee": ["front_right_knee", "right_front_knee", "rf_knee"],
        "left_hind_knee": ["back_left_knee", "left_hind_knee", "lh_knee"],
        "right_hind_knee": ["back_right_knee", "right_hind_knee", "rh_knee"],
        "left_hip": ["back_left_thai", "left_hip", "lh_hip"],
        "right_hip": ["back_right_thai", "right_hip", "rh_hip"],
    }

    FEATURE_NAMES = [
        "head_vertical_disp_mean", "head_vertical_disp_std",
        "spine_angle_mean", "spine_angle_std",
        "step_duration_left_mean", "step_duration_left_std",
        "step_duration_right_mean", "step_duration_right_std",
        "temporal_asymmetry_ratio", "step_frequency",
        "hip_sway_amplitude", "hip_sway_range",
        "stride_length_left_cv", "stride_length_right_cv",
        "knee_angle_asymmetry", "back_curvature_variance",
    ]

    def __init__(self, fps: float = 30.0, framework: str = "deeplabcut",
                 min_confidence: float = 0.3):
        self.fps = fps
        self.framework = framework
        self.min_conf = min_confidence
        self._keypoint_map = None

    @property
    def num_features(self) -> int:
        return len(self.FEATURE_NAMES)

    def _resolve_keypoints(self, columns) -> Dict[str, Optional[str]]:
        """Dynamically resolve keypoint names from CSV columns."""
        col_strs = [str(c).lower() for c in columns]
        resolved = {}
        for group_name, candidates in self.KEYPOINT_GROUPS.items():
            found = None
            for candidate in candidates:
                for cs in col_strs:
                    if candidate in cs:
                        found = candidate
                        break
                if found:
                    break
            resolved[group_name] = found
        return resolved

    def _get_keypoint_data(self, df: pd.DataFrame, kp_name: Optional[str]
                           ) -> Optional[np.ndarray]:
        """Get (x, y, confidence) for a keypoint. Returns (N, 3) or None."""
        if kp_name is None:
            return None
        matching = [c for c in df.columns if kp_name in str(c).lower()]
        if len(matching) < 3:
            return None
        try:
            x = pd.to_numeric(df[matching[0]], errors='coerce').values
            y = pd.to_numeric(df[matching[1]], errors='coerce').values
            c = pd.to_numeric(df[matching[2]], errors='coerce').values
            return np.column_stack([x, y, c])
        except Exception:
            return None

    def extract_from_csv(self, csv_path: str) -> np.ndarray:
        """Extract features from a DLC CSV file. Returns (16,) array (NaN = missing)."""
        try:
            if self.framework == "deeplabcut":
                df = pd.read_csv(csv_path, header=[0, 1, 2])
                new_cols = []
                for c in df.columns:
                    if isinstance(c, tuple) and len(c) >= 3:
                        # Use only bodypart (1) and coord (2), ignore scorer (0)
                        part = str(c[1])
                        coord = str(c[2])
                        new_cols.append(f"{part}_{coord}".lower())
                    else:
                        new_cols.append('_'.join(str(x) for x in c).lower())
                df.columns = new_cols
            else:
                df = pd.read_csv(csv_path, index_col=0)
                df.columns = [str(c).lower() for c in df.columns]

            # Resolve keypoints per CSV (handles different DLC schemas)
            self._keypoint_map = self._resolve_keypoints(df.columns)

            return self._compute_features(df)
        except Exception:
            return np.full(self.num_features, np.nan, dtype=np.float32)

    def _compute_features(self, df: pd.DataFrame) -> np.ndarray:
        """Compute all 16 features from parsed DataFrame. NaN = not computable."""
        feats = np.full(self.num_features, np.nan, dtype=np.float32)
        n_frames = len(df)

        if n_frames < 30:
            return feats

        km = self._keypoint_map

        # --- Head bob ---
        head = self._get_keypoint_data(df, km.get("head"))
        if head is not None:
            mask = head[:, 2] > self.min_conf
            if mask.sum() > 10:
                y = head[mask, 1]
                dy = np.diff(y)
                feats[0] = np.mean(np.abs(dy))
                feats[1] = np.std(dy)

        # --- Spine angle ---
        withers = self._get_keypoint_data(df, km.get("withers"))
        spine = self._get_keypoint_data(df, km.get("spine"))
        tail = self._get_keypoint_data(df, km.get("tail_base"))
        if all(v is not None for v in [withers, spine, tail]):
            angles = self._compute_angle_trajectory(withers, spine, tail)
            if len(angles) > 5:
                feats[2] = np.nanmean(angles)
                feats[3] = np.nanstd(angles)
                feats[15] = np.nanvar(angles)  # back_curvature_variance

        # --- Step timing (front hooves) ---
        lf = self._get_keypoint_data(df, km.get("left_front_hoof"))
        rf = self._get_keypoint_data(df, km.get("right_front_hoof"))
        steps_l = self._detect_steps(lf) if lf is not None else np.array([])
        steps_r = self._detect_steps(rf) if rf is not None else np.array([])

        if len(steps_l) > 1:
            dur_l = np.diff(steps_l) / self.fps
            feats[4] = np.median(dur_l)
            feats[5] = np.std(dur_l)
        if len(steps_r) > 1:
            dur_r = np.diff(steps_r) / self.fps
            feats[6] = np.median(dur_r)
            feats[7] = np.std(dur_r)

        # --- Temporal asymmetry ---
        if feats[4] > 0 and feats[6] > 0:
            feats[8] = abs(feats[4] - feats[6]) / max(feats[4], feats[6])

        # --- Step frequency ---
        total_steps = len(steps_l) + len(steps_r)
        duration_sec = n_frames / self.fps
        feats[9] = total_steps / max(duration_sec, 1.0)

        # --- Hip sway ---
        lh = self._get_keypoint_data(df, km.get("left_hip"))
        rh = self._get_keypoint_data(df, km.get("right_hip"))
        if lh is not None and rh is not None:
            mask = (lh[:, 2] > self.min_conf) & (rh[:, 2] > self.min_conf)
            if mask.sum() > 10:
                cx = (lh[mask, 0] + rh[mask, 0]) / 2
                feats[10] = np.std(cx)
                feats[11] = np.ptp(cx)

        # --- Stride length CV ---
        if lf is not None and len(steps_l) > 2:
            sl = np.abs(np.diff(lf[steps_l, 0]))
            feats[12] = np.std(sl) / (np.mean(sl) + 1e-6)
        if rf is not None and len(steps_r) > 2:
            sr = np.abs(np.diff(rf[steps_r, 0]))
            feats[13] = np.std(sr) / (np.mean(sr) + 1e-6)

        # --- Knee angle asymmetry (hind legs — most relevant for lameness) ---
        lhk = self._get_keypoint_data(df, km.get("left_hind_knee"))
        rhk = self._get_keypoint_data(df, km.get("right_hind_knee"))
        lhh = self._get_keypoint_data(df, km.get("left_hind_hoof"))
        rhh = self._get_keypoint_data(df, km.get("right_hind_hoof"))
        if all(v is not None for v in [lh, lhk, lhh]) and all(v is not None for v in [rh, rhk, rhh]):
            la = self._compute_angle_trajectory(lh, lhk, lhh)
            ra = self._compute_angle_trajectory(rh, rhk, rhh)
            if len(la) > 5 and len(ra) > 5:
                feats[14] = abs(np.nanmean(la) - np.nanmean(ra))

        feats = np.where(np.isfinite(feats), feats, np.nan)  # inf → NaN, keep NaN for missing
        return feats

    def _detect_steps(self, kp_data: np.ndarray) -> np.ndarray:
        """Detect heel strikes from vertical trajectory."""
        mask = kp_data[:, 2] > self.min_conf
        if mask.sum() < 15:
            return np.array([])
        y = np.where(mask, kp_data[:, 1], np.nan)
        nans = np.isnan(y)
        if nans.all():
            return np.array([])
        x_interp = np.arange(len(y))
        y[nans] = np.interp(x_interp[nans], x_interp[~nans], y[~nans])
        peaks, _ = find_peaks(y, distance=int(0.3 * self.fps), prominence=3)
        return peaks

    def _compute_angle_trajectory(self, p1: np.ndarray, p2: np.ndarray,
                                   p3: np.ndarray) -> np.ndarray:
        """Compute angle at p2 formed by p1-p2-p3 over frames."""
        n = min(len(p1), len(p2), len(p3))
        angles = []
        for i in range(n):
            if p1[i, 2] > self.min_conf and p2[i, 2] > self.min_conf and p3[i, 2] > self.min_conf:
                v1 = p1[i, :2] - p2[i, :2]
                v2 = p3[i, :2] - p2[i, :2]
                cos_a = np.dot(v1, v2) / (np.linalg.norm(v1) * np.linalg.norm(v2) + 1e-8)
                angles.append(np.degrees(np.arccos(np.clip(cos_a, -1, 1))))
            else:
                angles.append(np.nan)
        return np.array(angles)


# Initialize
pose_extractor = PoseFeatureExtractor(
    fps=30.0,
    framework=CFG["POSE_FRAMEWORK"],
    min_confidence=CFG["MIN_CONFIDENCE"]
)
print(f"✅ PoseFeatureExtractor initialized | {pose_extractor.num_features} features")
print(f"Feature names: {pose_extractor.FEATURE_NAMES}")


In [ ]:
# ============================================================
# Extract pose features for all videos (cached)
# ============================================================

def extract_all_pose_features(data_df: pd.DataFrame, extractor: PoseFeatureExtractor,
                               cache_path: str = None) -> np.ndarray:
    """Extract pose features for all videos. Uses cache if available."""
    if cache_path and os.path.exists(cache_path):
        feats = np.load(cache_path)
        print(f"✅ Loaded cached pose features: {feats.shape}")
        return feats

    print(f"🔄 Extracting pose features for {len(data_df)} videos...")
    all_feats = []
    missing = 0

    for idx, row in data_df.iterrows():
        csv_path = row.get("dlc_csv_path")
        if csv_path and os.path.exists(str(csv_path)):
            feat = extractor.extract_from_csv(str(csv_path))
        else:
            feat = np.full(extractor.num_features, np.nan, dtype=np.float32)
            missing += 1
        all_feats.append(feat)

        if (idx + 1) % 200 == 0:
            print(f"  Processed {idx+1}/{len(data_df)}...")

    feats = np.array(all_feats, dtype=np.float32)

    if cache_path:
        np.save(cache_path, feats)
        print(f"💾 Cached pose features to {cache_path}")

    print(f"✅ Pose features: {feats.shape} | Missing DLC: {missing}/{len(data_df)}")
    return feats

cache_path = os.path.join(CFG["RESULTS_DIR"], "pose_features_v2_cache.npy")
pose_features = extract_all_pose_features(data_df, pose_extractor, cache_path)


In [ ]:
# ============================================================
# Pose Feature Distribution Analysis
# ============================================================

def plot_pose_feature_distributions(features: np.ndarray, data_labels: np.ndarray,
                                     feature_names: list, save_path: str = None):
    """Box plots comparing healthy vs lame for each pose feature."""
    fig, axes = plt.subplots(4, 4, figsize=(20, 16))
    axes = axes.flatten()

    for i, (name, ax) in enumerate(zip(feature_names, axes)):
        healthy_vals = features[data_labels == 0, i]
        lame_vals = features[data_labels == 1, i]

        data = [healthy_vals[~np.isnan(healthy_vals)], lame_vals[~np.isnan(lame_vals)]]
        bp = ax.boxplot(data, labels=["Healthy", "Lame"], patch_artist=True,
                       boxprops=dict(alpha=0.7))
        bp['boxes'][0].set_facecolor('#2ecc71')
        bp['boxes'][1].set_facecolor('#e74c3c')

        if len(data[0]) > 5 and len(data[1]) > 5:
            t_stat, p_val = stats.ttest_ind(data[0], data[1], equal_var=False)
            sig = "***" if p_val < 0.001 else "**" if p_val < 0.01 else "*" if p_val < 0.05 else "ns"
            ax.set_title(f"{name}\n(p={p_val:.4f}) {sig}", fontsize=9)
        else:
            ax.set_title(name, fontsize=9)
        ax.tick_params(labelsize=8)

    plt.suptitle("Pose Feature Distributions: Healthy vs Lame", fontsize=14, fontweight='bold')
    plt.tight_layout()

    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
        print(f"💾 Saved: {save_path}")
    plt.show()

plot_pose_feature_distributions(
    pose_features, data_df["label"].values, pose_extractor.FEATURE_NAMES,
    save_path=os.path.join(CFG["RESULTS_DIR"], "pose_feature_distributions.png")
)


---
## Section 4: VideoMAE Encoder — Partial Fine-Tuning (Hybrid)

**Strategy (ADR-001 — Revised):**
- Blocks 0-9: **FROZEN** — intermediate features pre-computed once & cached (768-dim)
- Blocks 10-11: **TRAINABLE** — domain adapter, trained during each fold
- Projection: 768 → 256 (trainable)
- Hybrid approach: blocks 0-9 output cached (~28 MB), blocks 10-11 run live

> **Why hybrid?** Pre-computing frozen block outputs avoids redundant forward passes
> through 10 transformer layers every epoch. Only the 2 trainable blocks + projection
> + temporal model are computed per batch. ~10× faster than online inference.

**Architecture:**
```
[One-time pre-computation]
Raw clips → VideoMAE blocks 0-9 (frozen) → mean pool → cache (768-dim per clip)

[Per-epoch training]
Cached 768-dim → blocks 10-11 (trainable) → layernorm → projection → 256-dim
→ concat(visual_256, pose_16) → Temporal Transformer → BCE
```


In [ ]:
# ============================================================
# SECTION 4: VideoMAE Encoder — Partial Fine-Tuning (Hybrid)
# ============================================================

class VideoMAEFrozenEncoder(nn.Module):
    """
    Uses full VideoMAE model with output_hidden_states to extract
    intermediate features after block (split_at - 1).
    All parameters frozen; run once and cached.
    """

    def __init__(self, videomae_model, split_at: int = 10):
        super().__init__()
        self.model = videomae_model
        self.split_at = split_at
        for p in self.parameters():
            p.requires_grad = False

    @torch.no_grad()
    def forward(self, pixel_values: torch.Tensor) -> torch.Tensor:
        """
        Args:
            pixel_values: (B, T, C, H, W)
        Returns:
            intermediate_features: (B, 768) — mean-pooled after block (split_at-1)
        """
        outputs = self.model(pixel_values, output_hidden_states=True)
        # hidden_states tuple: (embedding_output, block0_out, ..., block11_out)
        # Index split_at → output after block (split_at - 1)
        intermediate = outputs.hidden_states[self.split_at]  # (B, seq_len, 768)
        return intermediate.mean(dim=1)  # (B, 768)


class VideoMAEDomainAdapter(nn.Module):
    """
    Domain adaptation FFN replacing VideoMAE blocks 10-11.

    With mean-pooled input (seq_length=1), self-attention degenerates
    to identity. Two residual FFN blocks replicate the blocks' capacity
    (768 → 3072 → 768 each), followed by projection (768 → 256).
    """

    def __init__(self, input_dim: int = 768, hidden_dim: int = 3072,
                 projection_dim: int = 256, dropout: float = 0.1):
        super().__init__()

        # Block 1 FFN equivalent (residual)
        self.ffn1 = nn.Sequential(
            nn.LayerNorm(input_dim),
            nn.Linear(input_dim, hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, input_dim),
            nn.Dropout(dropout),
        )

        # Block 2 FFN equivalent (residual)
        self.ffn2 = nn.Sequential(
            nn.LayerNorm(input_dim),
            nn.Linear(input_dim, hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, input_dim),
            nn.Dropout(dropout),
        )

        # Projection: 768 → projection_dim
        self.projection = nn.Sequential(
            nn.LayerNorm(input_dim),
            nn.Linear(input_dim, projection_dim),
            nn.GELU(),
            nn.Dropout(dropout),
        )

        n_ffn = sum(p.numel() for p in self.ffn1.parameters()) +                 sum(p.numel() for p in self.ffn2.parameters())
        n_proj = sum(p.numel() for p in self.projection.parameters())
        print(f"  Domain adapter FFN: {n_ffn:,} params")
        print(f"  Projection: {n_proj:,} params")
        print(f"  Total trainable adapter: {n_ffn + n_proj:,} params")

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Args:
            x: (B, 768) — mean-pooled intermediate features from frozen blocks
        Returns:
            adapted: (B, projection_dim) — domain-adapted visual embeddings
        """
        x = x + self.ffn1(x)    # Residual block 1
        x = x + self.ffn2(x)    # Residual block 2
        return self.projection(x)  # (B, projection_dim)


# ── Load VideoMAE & create components ──
print("📦 Loading VideoMAE for Partial Fine-Tuning (hybrid)...")
_full_videomae = VideoMAEModel.from_pretrained(CFG["VIDEOMAE_MODEL"])

split_at = min(CFG["TRAINABLE_BLOCKS"])

# Domain adapter — standalone FFN (no VideoMAE block dependency)
domain_adapter = VideoMAEDomainAdapter(
    input_dim=CFG["VIDEOMAE_DIM"],
    projection_dim=CFG["PROJECTION_DIM"],
).to(DEVICE)

# Frozen encoder wraps full model (uses official forward + output_hidden_states)
frozen_encoder = VideoMAEFrozenEncoder(_full_videomae, split_at=split_at).to(DEVICE)

# Verify freeze status
n_frozen = sum(p.numel() for p in frozen_encoder.parameters() if not p.requires_grad)
n_trainable_enc = sum(p.numel() for p in frozen_encoder.parameters() if p.requires_grad)
n_trainable_adapt = sum(p.numel() for p in domain_adapter.parameters() if p.requires_grad)
print(f"\n✅ VideoMAE hybrid setup complete:")
print(f"   Frozen encoder: {n_frozen:,} params (ALL FROZEN)")
print(f"   Domain adapter: {n_trainable_adapt:,} trainable params")
assert n_trainable_enc == 0, "❌ Frozen encoder has trainable params!"
print("🗑️ Full model will be freed after pre-computation")


In [ ]:
# ============================================================
# Clip Extraction & Encoding (standalone functions)
# ============================================================

def extract_clips_from_video(video_path: str, cfg: dict) -> Optional[List[np.ndarray]]:
    """Extract clips from a single video. Each clip: (CLIP_LENGTH, H, W, 3)."""
    try:
        cap = cv2.VideoCapture(video_path)
        if not cap.isOpened():
            return None

        frames = []
        while True:
            ret, frame = cap.read()
            if not ret:
                break
            frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            frame = cv2.resize(frame, (cfg["IMG_SIZE"], cfg["IMG_SIZE"]))
            frames.append(frame)
        cap.release()

        if len(frames) < cfg["CLIP_LENGTH"]:
            return None

        clips = []
        stride = max(1, (len(frames) - cfg["CLIP_LENGTH"]) // cfg["NUM_CLIPS"])
        for start in range(0, len(frames) - cfg["CLIP_LENGTH"] + 1, stride):
            clip = np.array(frames[start:start + cfg["CLIP_LENGTH"]])
            clips.append(clip)
            if len(clips) >= cfg["NUM_CLIPS"] * 2:
                break

        return clips if clips else None
    except Exception:
        return None


@torch.no_grad()
def encode_clips_intermediate(encoder: VideoMAEFrozenEncoder, clips: List[np.ndarray],
                               device: str) -> np.ndarray:
    """Encode clips with frozen blocks 0-9. Returns (N, 768) numpy."""
    encoder.eval()
    embeddings = []

    for clip in clips:
        # Normalize with ImageNet stats
        clip_f = clip.astype(np.float32) / 255.0
        mean = np.array([0.485, 0.456, 0.406])
        std = np.array([0.229, 0.224, 0.225])
        clip_f = (clip_f - mean) / std

        # (T, H, W, C) → (1, T, C, H, W) — HuggingFace VideoMAE format
        tensor = torch.from_numpy(clip_f).permute(0, 3, 1, 2).unsqueeze(0).float()
        tensor = tensor.to(device)

        emb = encoder(tensor)  # (1, 768)
        embeddings.append(emb.cpu().numpy().squeeze())

    return np.array(embeddings)


print("✅ Clip extraction & encoding functions defined")


In [ ]:
# ============================================================
# Pre-compute intermediate features (blocks 0-9, cached)
# ============================================================

def precompute_intermediate_features(video_paths, encoder, cfg, device,
                                      cache_path=None):
    """
    Pre-compute frozen blocks 0-9 output for all videos.
    Returns list of (n_clips_i, 768) numpy arrays.
    """
    if cache_path and os.path.exists(cache_path):
        data = np.load(cache_path, allow_pickle=True).item()
        print(f"✅ Loaded cached intermediate features: {len(data['features'])} videos")
        return data['features']

    print(f"🔄 Pre-computing intermediate features (blocks 0-{min(cfg['TRAINABLE_BLOCKS'])-1}) "
          f"for {len(video_paths)} videos...")
    all_features = []
    n_failed = 0

    for i, vpath in enumerate(video_paths):
        clips = extract_clips_from_video(vpath, cfg)
        if clips is not None and len(clips) > 0:
            feats = encode_clips_intermediate(encoder, clips, device)
            all_features.append(feats)
        else:
            all_features.append(np.zeros((0, cfg["VIDEOMAE_DIM"]), dtype=np.float32))
            n_failed += 1

        if (i + 1) % 50 == 0:
            print(f"  {i+1}/{len(video_paths)} videos...")

    if cache_path:
        np.save(cache_path, {'features': all_features}, allow_pickle=True)
        print(f"💾 Cached intermediate features")

    n_valid = len(video_paths) - n_failed
    print(f"✅ Intermediate features: {n_valid}/{len(video_paths)} videos encoded")
    return all_features


vis_cache = os.path.join(CFG["RESULTS_DIR"], "intermediate_features_cache.npy")
all_intermediate_features = precompute_intermediate_features(
    data_df["video_path"].values, frozen_encoder, CFG, DEVICE,
    cache_path=vis_cache
)

# Free frozen encoder (and the full model it wraps) — no longer needed
del frozen_encoder
torch.cuda.empty_cache()
print("🗑️ Frozen encoder + full VideoMAE model released from GPU memory")
print(f"📊 Domain adapter remains on {DEVICE} for training")


---
## Sections 5-6: Temporal Transformer + Classification Head

**Pipeline per video (training):**
```
cached intermediate (768-dim per clip)
→ Domain Adapter (blocks 10-11, trainable) → projection → 256-dim
→ concat(visual_256, pose_16) → 272-dim
→ Linear projection → 256-dim
→ Positional Encoding
→ TransformerEncoder (4 layers, 8 heads, causal mask)
→ Mean pooling → 256-dim
→ FC → Sigmoid → p(lame) ∈ [0, 1]
```

> **Two-LR training:** Domain adapter at `LR_VIDEOMAE` (1e-5),
> temporal model at `LR_HEAD` (1e-4).


In [ ]:
# ============================================================
# SECTIONS 5-6: Temporal Transformer + Classification Head
# ============================================================

class CowLamenessModelV32(nn.Module):
    """
    Complete lameness detection model with domain adaptation.

    Input: sequence of pre-computed intermediate clip embeddings (768-dim) + pose
    Output: binary probability p(lame)

    Architecture:
        1. Domain adapter: blocks 10-11 (768 → 768) → projection (768 → 256)
        2. Concat with pose: (256 + 16) = 272
        3. Input projection: 272 → hidden_dim
        4. Positional encoding (sinusoidal)
        5. TransformerEncoder with causal mask (4 layers, 8 heads)
        6. Temporal mean pooling
        7. Classification head → sigmoid
    """

    def __init__(self, adapter: VideoMAEDomainAdapter, pose_dim: int,
                 hidden_dim: int, num_heads: int, num_layers: int,
                 dropout: float, max_clips: int = 32):
        super().__init__()

        self.adapter = adapter
        visual_dim = adapter.projection[1].out_features  # nn.Linear(768, projection_dim)
        self.input_dim = visual_dim + pose_dim
        self.hidden_dim = hidden_dim

        # Input projection: (visual + pose) → hidden_dim
        self.input_proj = nn.Sequential(
            nn.Linear(self.input_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
        )

        # Sinusoidal positional encoding
        self.register_buffer('pos_encoding',
                             self._create_pos_encoding(hidden_dim, max_clips))

        # Transformer encoder
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=hidden_dim,
            nhead=num_heads,
            dim_feedforward=hidden_dim * 4,
            dropout=dropout,
            activation='gelu',
            batch_first=True,
            norm_first=True,
        )
        self.transformer = nn.TransformerEncoder(encoder_layer,
                                                  num_layers=num_layers)

        # Classification head
        self.classifier = nn.Sequential(
            nn.LayerNorm(hidden_dim),
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim // 2, 1),
        )

    def _create_pos_encoding(self, d_model: int, max_len: int) -> torch.Tensor:
        pe = torch.zeros(max_len, d_model)
        pos = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div = torch.exp(torch.arange(0, d_model, 2).float() *
                        (-np.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)
        return pe.unsqueeze(0)  # (1, max_len, d_model)

    def _get_causal_mask(self, seq_len: int, device: torch.device) -> torch.Tensor:
        """Upper triangular causal mask (True = masked)."""
        return torch.triu(torch.ones(seq_len, seq_len, device=device),
                          diagonal=1).bool()

    def forward(self, clip_intermediate: torch.Tensor, clip_pose: torch.Tensor,
                padding_mask: Optional[torch.Tensor] = None,
                use_causal: bool = True) -> Tuple[torch.Tensor, torch.Tensor]:
        """
        Args:
            clip_intermediate: (B, N, 768) — cached blocks 0-9 output
            clip_pose:         (B, N, pose_dim) — DLC pose features
            padding_mask:      (B, N) — True where padded
            use_causal:        whether to apply causal attention mask

        Returns:
            logits: (B, 1)
            attention_weights: (B, N) — temporal importance proxy
        """
        B, N, D = clip_intermediate.shape

        # Domain adapter: blocks 10-11 + projection per clip
        flat = clip_intermediate.reshape(B * N, D)  # (B*N, 768)
        clip_visual = self.adapter(flat)  # (B*N, projection_dim)
        clip_visual = clip_visual.reshape(B, N, -1)  # (B, N, 256)

        # Concatenate visual + pose
        x = torch.cat([clip_visual, clip_pose], dim=-1)  # (B, N, 272)

        # Project to hidden dim
        x = self.input_proj(x)  # (B, N, hidden)

        # Add positional encoding
        x = x + self.pos_encoding[:, :N, :]

        # Causal mask
        causal_mask = self._get_causal_mask(N, x.device) if use_causal else None

        # Transformer
        x = self.transformer(x, mask=causal_mask,
                            src_key_padding_mask=padding_mask)  # (B, N, hidden)

        # Temporal pooling (mean over non-padded positions)
        if padding_mask is not None:
            valid_mask = ~padding_mask  # (B, N)
            x_masked = x * valid_mask.unsqueeze(-1).float()
            pooled = x_masked.sum(dim=1) / valid_mask.sum(
                dim=1, keepdim=True).float().clamp(min=1)
        else:
            pooled = x.mean(dim=1)

        # Attention weights proxy for interpretability
        with torch.no_grad():
            attn_weights = torch.norm(x, dim=-1)  # (B, N)
            if padding_mask is not None:
                attn_weights = attn_weights.masked_fill(padding_mask, 0.0)
            attn_weights = F.softmax(attn_weights, dim=-1)

        logits = self.classifier(pooled)  # (B, 1)
        return logits, attn_weights


# ── Initialize model ──
model = CowLamenessModelV32(
    adapter=domain_adapter,
    pose_dim=CFG["POSE_FEAT_DIM"],
    hidden_dim=CFG["HIDDEN_DIM"],
    num_heads=CFG["NUM_HEADS"],
    num_layers=CFG["NUM_LAYERS"],
    dropout=CFG["DROPOUT"],
    max_clips=CFG["NUM_CLIPS"] * 2,
).to(DEVICE)

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
adapter_params = sum(p.numel() for p in model.adapter.parameters() if p.requires_grad)
temporal_params = trainable_params - adapter_params
print(f"\n✅ CowLamenessModelV32 (Partial FT)")
print(f"   Total: {total_params:,} | Trainable: {trainable_params:,}")
print(f"   Adapter (blocks 10-11 + proj): {adapter_params:,} @ LR={CFG['LR_VIDEOMAE']}")
print(f"   Temporal + classifier: {temporal_params:,} @ LR={CFG['LR_HEAD']}")

# Quick shape test
with torch.no_grad():
    dummy_int = torch.randn(2, 8, CFG["VIDEOMAE_DIM"]).to(DEVICE)
    dummy_pose = torch.randn(2, 8, CFG["POSE_FEAT_DIM"]).to(DEVICE)
    out, attn = model(dummy_int, dummy_pose)
    assert out.shape == (2, 1), f"Expected (2,1) got {out.shape}"
    assert attn.shape == (2, 8), f"Expected (2,8) got {attn.shape}"
    print(f"✅ Shape test passed: output={out.shape}, attention={attn.shape}")

# Free shape-test model (fold models are created fresh per fold)
del model
torch.cuda.empty_cache()


---
## Section 7: Dataset & DataLoader

Each sample uses **hybrid** features:
- `clip_intermediate ∈ ℝ^(N×768)` — cached blocks 0-9 output (frozen)
- `pose_feat ∈ ℝ^16` — DLC pose features (replicated per clip)
- During training, blocks 10-11 (trainable) adapt the 768-dim features


In [ ]:
# ============================================================
# SECTION 7: Dataset & DataLoader (intermediate features)
# ============================================================

class CowLamenessDatasetV32(Dataset):
    """
    Dataset using pre-computed intermediate features (blocks 0-9 output).

    Blocks 10-11 (domain adapter) run online during training.
    """

    def __init__(self, intermediate_features_list: list, pose_features: np.ndarray,
                 labels: np.ndarray, cfg: dict):
        self.intermediate_features = intermediate_features_list  # list of (n_clips_i, 768)
        self.pose_features = pose_features  # (N_samples, 16)
        self.labels = labels
        self.cfg = cfg

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx: int):
        vis = self.intermediate_features[idx]   # (n_clips, 768) or (0, 768)
        pose = np.nan_to_num(self.pose_features[idx], nan=0.0)  # NaN → 0 for model input
        label = self.labels[idx]

        n_clips = len(vis)
        target_n = self.cfg["NUM_CLIPS"]
        mask = np.zeros(target_n, dtype=bool)

        # Replicate pose features for each clip
        if n_clips > 0:
            pose_rep = np.tile(pose, (n_clips, 1))
        else:
            vis = np.zeros((0, self.cfg["VIDEOMAE_DIM"]), dtype=np.float32)
            pose_rep = np.zeros((0, self.cfg["POSE_FEAT_DIM"]), dtype=np.float32)

        # Pad/truncate to target_n
        if n_clips >= target_n:
            indices = np.linspace(0, n_clips - 1, target_n, dtype=int)
            vis = vis[indices]
            pose_rep = pose_rep[indices]
        elif n_clips > 0:
            pad_v = np.zeros((target_n - n_clips, self.cfg["VIDEOMAE_DIM"]),
                             dtype=np.float32)
            pad_p = np.zeros((target_n - n_clips, self.cfg["POSE_FEAT_DIM"]),
                             dtype=np.float32)
            vis = np.concatenate([vis, pad_v], axis=0)
            pose_rep = np.concatenate([pose_rep, pad_p], axis=0)
            mask[n_clips:] = True
        else:
            vis = np.zeros((target_n, self.cfg["VIDEOMAE_DIM"]), dtype=np.float32)
            pose_rep = np.zeros((target_n, self.cfg["POSE_FEAT_DIM"]),
                                dtype=np.float32)
            mask[:] = True

        return vis, pose_rep, label, mask


def collate_fn(batch):
    """Custom collate: stack arrays and convert to tensors."""
    visuals, poses, labels, masks = zip(*batch)
    return (
        torch.tensor(np.array(visuals), dtype=torch.float32),
        torch.tensor(np.array(poses), dtype=torch.float32),
        torch.tensor(np.array(labels), dtype=torch.long),
        torch.tensor(np.array(masks), dtype=torch.bool),
    )

print("✅ Dataset & DataLoader classes defined")


---
## Section 8: Training Loop

- **5-fold subject-level cross-validation** (StratifiedGroupKFold)
- **BCE loss** with class weights for imbalance
- **AdamW** with **2 LR groups**: domain adapter (1e-5), temporal model (1e-4)
- **ReduceLROnPlateau** scheduler
- **Early stopping** on validation loss (patience=7)
- **Gradient clipping** (max_norm=1.0)

> Blocks 10-11 of VideoMAE are fine-tuned at lower LR for domain adaptation.
> Temporal Transformer + classifier are trained at standard LR.


In [ ]:
# ============================================================
# SECTION 8: Training Functions
# ============================================================

def train_one_epoch(model, dataloader, optimizer, criterion, device, cfg):
    """Train for one epoch. Returns mean loss."""
    model.train()
    total_loss = 0
    n_batches = 0

    for visuals, poses, labels, masks in dataloader:
        visuals = visuals.to(device)
        poses = poses.to(device)
        labels = labels.float().to(device)
        masks = masks.to(device)

        optimizer.zero_grad()
        logits, _ = model(visuals, poses, padding_mask=masks)
        loss = criterion(logits.squeeze(-1), labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), cfg["GRAD_CLIP"])
        optimizer.step()

        total_loss += loss.item()
        n_batches += 1

    return total_loss / max(n_batches, 1)


@torch.no_grad()
def evaluate(model, dataloader, criterion, device, use_causal=True):
    """Evaluate model. Returns metrics dict, probs, labels, attention."""
    model.eval()

    total_loss = 0
    all_probs, all_labels, all_attns = [], [], []
    n_batches = 0

    for visuals, poses, labels, masks in dataloader:
        visuals = visuals.to(device)
        poses = poses.to(device)
        labels_f = labels.float().to(device)
        masks = masks.to(device)

        logits, attn = model(visuals, poses, padding_mask=masks, use_causal=use_causal)
        loss = criterion(logits.squeeze(-1), labels_f)

        probs = torch.sigmoid(logits.squeeze(-1))
        all_probs.extend(probs.cpu().numpy())
        all_labels.extend(labels.numpy())
        all_attns.extend(attn.cpu().numpy())

        total_loss += loss.item()
        n_batches += 1

    avg_loss = total_loss / max(n_batches, 1)
    probs_arr = np.array(all_probs)
    labels_arr = np.array(all_labels)
    preds_arr = (probs_arr >= 0.5).astype(int)

    metrics = {
        "loss": avg_loss,
        "accuracy": accuracy_score(labels_arr, preds_arr),
        "precision": precision_score(labels_arr, preds_arr, zero_division=0),
        "recall": recall_score(labels_arr, preds_arr, zero_division=0),
        "f1": f1_score(labels_arr, preds_arr, zero_division=0),
        "auc": roc_auc_score(labels_arr, probs_arr) if len(np.unique(labels_arr)) > 1 else 0.0,
    }

    return metrics, probs_arr, labels_arr, np.array(all_attns)

print("✅ Training functions defined")


In [ ]:
# ============================================================
# SECTION 8: 5-Fold Cross-Validation Training
# ============================================================

def run_cross_validation(data_df, all_intermediate_features, pose_features, cfg, device):
    """Run 5-fold subject-level stratified CV with Partial FT."""
    video_paths = data_df["video_path"].values
    data_labels = data_df["label"].values
    animal_ids = data_df["animal_id"].values

    # Class weights for BCE
    n_healthy = (data_labels == 0).sum()
    n_lame = (data_labels == 1).sum()
    pos_weight = torch.tensor([n_healthy / n_lame]).to(device)
    print(f"📊 Class balance — Healthy: {n_healthy}, Lame: {n_lame}, "
          f"pos_weight: {pos_weight.item():.3f}")

    # CV splitter
    cv = StratifiedGroupKFold(n_splits=cfg["CV_FOLDS"], shuffle=True,
                               random_state=cfg["SEED"])

    fold_results = []
    all_fold_probs = []
    all_fold_labels = []
    best_models = []

    for fold, (train_idx, val_idx) in enumerate(
            cv.split(video_paths, data_labels, animal_ids)):
        print(f"\n{'='*60}")
        print(f"FOLD {fold+1}/{cfg['CV_FOLDS']}")
        print(f"{'='*60}")

        # Verify no animal leakage
        train_animals = set(animal_ids[train_idx])
        val_animals = set(animal_ids[val_idx])
        assert len(train_animals & val_animals) == 0, "❌ Animal leakage detected!"
        print(f"✅ No leakage | Train: {len(train_idx)} ({len(train_animals)} animals) | "
              f"Val: {len(val_idx)} ({len(val_animals)} animals)")

        # Create datasets with intermediate features
        train_vis = [all_intermediate_features[i] for i in train_idx]
        val_vis = [all_intermediate_features[i] for i in val_idx]

        train_ds = CowLamenessDatasetV32(
            train_vis, pose_features[train_idx], data_labels[train_idx], cfg
        )
        val_ds = CowLamenessDatasetV32(
            val_vis, pose_features[val_idx], data_labels[val_idx], cfg
        )

        train_loader = DataLoader(train_ds, batch_size=cfg["BATCH_SIZE"],
                                   shuffle=True, collate_fn=collate_fn, num_workers=0)
        val_loader = DataLoader(val_ds, batch_size=cfg["BATCH_SIZE"],
                                 shuffle=False, collate_fn=collate_fn, num_workers=0)

        # Fresh model for each fold (domain adapter re-initialized from pretrained)
        # Note: domain_adapter is shared reference — we need fresh copies per fold
        # Fresh adapter per fold via deepcopy
        import copy
        fold_adapter = copy.deepcopy(domain_adapter)

        fold_model = CowLamenessModelV32(
            adapter=fold_adapter,
            pose_dim=cfg["POSE_FEAT_DIM"],
            hidden_dim=cfg["HIDDEN_DIM"],
            num_heads=cfg["NUM_HEADS"],
            num_layers=cfg["NUM_LAYERS"],
            dropout=cfg["DROPOUT"],
        ).to(device)

        # Optimizer — 2 LR groups
        adapter_params = list(fold_model.adapter.parameters())
        temporal_params = [p for n, p in fold_model.named_parameters()
                          if not n.startswith("adapter.")]

        optimizer = torch.optim.AdamW([
            {"params": adapter_params, "lr": cfg["LR_VIDEOMAE"]},
            {"params": temporal_params, "lr": cfg["LR_HEAD"]},
        ], weight_decay=cfg["WEIGHT_DECAY"])

        criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
            optimizer, mode='min', factor=0.5, patience=3
        )

        # Training loop
        best_val_loss = float('inf')
        patience_counter = 0
        best_state = None
        history = {"train_loss": [], "val_loss": [], "val_acc": [],
                   "val_f1": [], "val_auc": []}

        for epoch in range(cfg["EPOCHS"]):
            train_loss = train_one_epoch(fold_model, train_loader,
                                         optimizer, criterion, device, cfg)
            val_metrics, val_probs, val_labels, _ = evaluate(
                fold_model, val_loader, criterion, device)

            history["train_loss"].append(train_loss)
            history["val_loss"].append(val_metrics["loss"])
            history["val_acc"].append(val_metrics["accuracy"])
            history["val_f1"].append(val_metrics["f1"])
            history["val_auc"].append(val_metrics["auc"])

            scheduler.step(val_metrics["loss"])

            # Early stopping
            if val_metrics["loss"] < best_val_loss - 0.001:
                best_val_loss = val_metrics["loss"]
                patience_counter = 0
                best_state = {
                    "model": {k: v.cpu().clone()
                              for k, v in fold_model.state_dict().items()},
                    "epoch": epoch,
                    "val_metrics": val_metrics,
                }
            else:
                patience_counter += 1

            if (epoch + 1) % 5 == 0 or patience_counter == 0:
                print(f"  Epoch {epoch+1:3d} | Train: {train_loss:.4f} | "
                      f"Val: {val_metrics['loss']:.4f} | "
                      f"Acc: {val_metrics['accuracy']:.3f} | "
                      f"F1: {val_metrics['f1']:.3f} | "
                      f"AUC: {val_metrics['auc']:.3f}"
                      + (" ★" if patience_counter == 0 else ""))

            if patience_counter >= cfg["PATIENCE"]:
                print(f"  ⏹ Early stopping at epoch {epoch+1}")
                break

        # Load best model and final evaluation
        fold_model.load_state_dict(best_state["model"])
        final_metrics, final_probs, final_labels, final_attns = evaluate(
            fold_model, val_loader, criterion, device)

        fold_results.append({
            "fold": fold + 1,
            "best_epoch": best_state["epoch"] + 1,
            "history": history,
            "fold_probs": final_probs,
            "fold_labels": final_labels,
            "fold_attns": final_attns,
            **final_metrics,
        })
        all_fold_probs.extend(final_probs)
        all_fold_labels.extend(final_labels)
        best_models.append(best_state)

        print(f"\n📊 Fold {fold+1} Best (epoch {best_state['epoch']+1}):")
        print(f"   Acc: {final_metrics['accuracy']:.4f} | "
              f"F1: {final_metrics['f1']:.4f} | AUC: {final_metrics['auc']:.4f}")

        # Free GPU memory after each fold
        del fold_model, fold_adapter
        torch.cuda.empty_cache()

    return fold_results, np.array(all_fold_probs), np.array(all_fold_labels), best_models

# ═══════════════ RUN TRAINING ═══════════════
print("\n🚀 Starting 5-Fold Cross-Validation Training (Partial FT)...")
fold_results, all_probs, all_labels, best_models = run_cross_validation(
    data_df, all_intermediate_features, pose_features, CFG, DEVICE
)
print("\n✅ Training complete!")


---
## Section 9: Comprehensive Evaluation (Q1 Journal Standard)

- Confusion matrix (counts + normalized)
- ROC curve (**per-fold** + mean)
- Precision-Recall curve (**per-fold** + mean)
- Per-fold metrics table
- Learning curves
- Statistical significance test + 95% CI


In [ ]:
# ============================================================
# SECTION 9: Confusion Matrix
# ============================================================

def plot_confusion_matrix(true_labels, pred_probs, save_path=None):
    """Normalized confusion matrix heatmap."""
    preds = (pred_probs >= 0.5).astype(int)
    cm = confusion_matrix(true_labels, preds)
    cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    for ax, data, title, fmt in [(axes[0], cm, "Confusion Matrix (Counts)", "d"),
                                  (axes[1], cm_norm, "Confusion Matrix (Normalized)", ".2%")]:
        sns.heatmap(data, annot=True, fmt=fmt, cmap="Blues", ax=ax,
                   xticklabels=["Healthy", "Lame"], yticklabels=["Healthy", "Lame"],
                   cbar_kws={"shrink": 0.8})
        ax.set_xlabel("Predicted", fontsize=12)
        ax.set_ylabel("Actual", fontsize=12)
        ax.set_title(title, fontsize=13, fontweight="bold")

    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=200, bbox_inches='tight')
    plt.show()

plot_confusion_matrix(all_labels, all_probs,
    save_path=os.path.join(CFG["RESULTS_DIR"], "confusion_matrix.png"))


In [ ]:
# ============================================================
# ROC & PR Curves (per-fold + mean)
# ============================================================

def plot_roc_and_pr_curves(fold_results, agg_labels, agg_probs, save_path=None):
    """ROC and Precision-Recall curves with per-fold detail."""
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))

    # --- ROC ---
    ax = axes[0]
    for r in fold_results:
        if "fold_probs" in r and "fold_labels" in r:
            fl = r["fold_labels"]
            fp = r["fold_probs"]
            if len(np.unique(fl)) > 1:
                fpr_f, tpr_f, _ = roc_curve(fl, fp)
                auc_f = roc_auc_score(fl, fp)
                ax.plot(fpr_f, tpr_f, '--', alpha=0.35, linewidth=1,
                       label=f'Fold {r["fold"]} ({auc_f:.3f})')

    fpr, tpr, _ = roc_curve(agg_labels, agg_probs)
    auc_val = roc_auc_score(agg_labels, agg_probs)
    ax.plot(fpr, tpr, 'b-', linewidth=2.5, label=f'Mean (AUC = {auc_val:.3f})')
    ax.plot([0, 1], [0, 1], 'k--', alpha=0.3)
    ax.fill_between(fpr, tpr, alpha=0.08, color='blue')
    ax.set_xlabel('False Positive Rate', fontsize=12)
    ax.set_ylabel('True Positive Rate', fontsize=12)
    ax.set_title('ROC Curve', fontsize=13, fontweight='bold')
    ax.legend(fontsize=9, loc='lower right')
    ax.grid(True, alpha=0.3)

    # --- PR (per-fold + mean) ---
    ax = axes[1]
    for r in fold_results:
        if "fold_probs" in r and "fold_labels" in r:
            fl = r["fold_labels"]
            fp = r["fold_probs"]
            if len(np.unique(fl)) > 1:
                prec_f, rec_f, _ = precision_recall_curve(fl, fp)
                ap_f = average_precision_score(fl, fp)
                ax.plot(rec_f, prec_f, '--', alpha=0.35, linewidth=1,
                       label=f'Fold {r["fold"]} (AP={ap_f:.3f})')

    prec, rec, _ = precision_recall_curve(agg_labels, agg_probs)
    ap = average_precision_score(agg_labels, agg_probs)
    ax.plot(rec, prec, 'r-', linewidth=2.5, label=f'Mean (AP = {ap:.3f})')
    ax.fill_between(rec, prec, alpha=0.1, color='red')
    ax.set_xlabel('Recall', fontsize=12)
    ax.set_ylabel('Precision', fontsize=12)
    ax.set_title('Precision-Recall Curve', fontsize=13, fontweight='bold')
    ax.legend(fontsize=9, loc='lower left')
    ax.grid(True, alpha=0.3)

    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=200, bbox_inches='tight')
    plt.show()

plot_roc_and_pr_curves(fold_results, all_labels, all_probs,
    save_path=os.path.join(CFG["RESULTS_DIR"], "roc_pr_curves.png"))


In [ ]:
# ============================================================
# Per-fold Results Table
# ============================================================

def print_fold_results_table(fold_results):
    """Print and display per-fold metrics as a table."""
    rows = []
    for r in fold_results:
        rows.append({
            "Fold": r["fold"],
            "Best Epoch": r["best_epoch"],
            "Accuracy": f"{r['accuracy']:.4f}",
            "Precision": f"{r['precision']:.4f}",
            "Recall": f"{r['recall']:.4f}",
            "F1": f"{r['f1']:.4f}",
            "AUC": f"{r['auc']:.4f}",
        })

    df = pd.DataFrame(rows)

    # Compute mean ± std
    metric_cols = ["accuracy", "precision", "recall", "f1", "auc"]
    means = {col: np.mean([r[col] for r in fold_results]) for col in metric_cols}
    stds = {col: np.std([r[col] for r in fold_results]) for col in metric_cols}

    summary_row = {
        "Fold": "Mean±Std",
        "Best Epoch": "-",
    }
    for col in metric_cols:
        key = col.capitalize() if col != "auc" else "AUC"
        summary_row[key] = f"{means[col]:.4f}±{stds[col]:.4f}"

    df = pd.concat([df, pd.DataFrame([summary_row])], ignore_index=True)

    print("\n" + "="*80)
    print("📊 5-FOLD CROSS-VALIDATION RESULTS")
    print("="*80)
    print(df.to_string(index=False))
    print("="*80)

    save_path = os.path.join(CFG["RESULTS_DIR"], "fold_results.csv")
    df.to_csv(save_path, index=False)
    print(f"💾 Saved to {save_path}")

    return df, means, stds

results_df, means, stds = print_fold_results_table(fold_results)


In [ ]:
# ============================================================
# Learning Curves (all folds)
# ============================================================

def plot_learning_curves(fold_results, save_path=None):
    """Training/validation loss and metrics over epochs for each fold."""
    n_folds = len(fold_results)
    fig, axes = plt.subplots(n_folds, 3, figsize=(18, 4 * n_folds))
    if n_folds == 1:
        axes = axes.reshape(1, -1)

    for i, r in enumerate(fold_results):
        h = r["history"]
        epochs = range(1, len(h["train_loss"]) + 1)

        # Loss
        axes[i, 0].plot(epochs, h["train_loss"], 'b-', label='Train')
        axes[i, 0].plot(epochs, h["val_loss"], 'r-', label='Val')
        axes[i, 0].set_title(f'Fold {r["fold"]} — Loss')
        axes[i, 0].legend()
        axes[i, 0].grid(True, alpha=0.3)

        # Accuracy
        axes[i, 1].plot(epochs, h["val_acc"], 'g-', label='Val Acc')
        axes[i, 1].set_title(f'Fold {r["fold"]} — Accuracy')
        axes[i, 1].set_ylim(0, 1)
        axes[i, 1].legend()
        axes[i, 1].grid(True, alpha=0.3)

        # AUC
        axes[i, 2].plot(epochs, h["val_auc"], 'm-', label='Val AUC')
        axes[i, 2].set_title(f'Fold {r["fold"]} — AUC')
        axes[i, 2].set_ylim(0, 1)
        axes[i, 2].legend()
        axes[i, 2].grid(True, alpha=0.3)

    plt.suptitle("Learning Curves (5-Fold CV)", fontsize=14, fontweight='bold')
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.show()

plot_learning_curves(fold_results,
    save_path=os.path.join(CFG["RESULTS_DIR"], "learning_curves.png"))


In [ ]:
# ============================================================
# Statistical Significance
# ============================================================

print("\n📊 Statistical Significance Analysis")
print("="*50)

# Classification report
preds = (all_probs >= 0.5).astype(int)
print("\nClassification Report:")
print(classification_report(all_labels, preds, target_names=["Healthy", "Lame"]))

# Paired t-test across folds (accuracy vs chance=0.5)
fold_accs = [r["accuracy"] for r in fold_results]
fold_aucs = [r["auc"] for r in fold_results]

t_acc, p_acc = stats.ttest_1samp(fold_accs, 0.5)
t_auc, p_auc = stats.ttest_1samp(fold_aucs, 0.5)
print(f"Accuracy vs chance (0.5): t={t_acc:.3f}, p={p_acc:.6f} "
      f"{'***' if p_acc<0.001 else '**' if p_acc<0.01 else '*' if p_acc<0.05 else 'ns'}")
print(f"AUC vs chance (0.5):      t={t_auc:.3f}, p={p_auc:.6f} "
      f"{'***' if p_auc<0.001 else '**' if p_auc<0.01 else '*' if p_auc<0.05 else 'ns'}")

# 95% CI
ci_acc = stats.t.interval(0.95, len(fold_accs)-1, loc=np.mean(fold_accs), scale=stats.sem(fold_accs))
ci_auc = stats.t.interval(0.95, len(fold_aucs)-1, loc=np.mean(fold_aucs), scale=stats.sem(fold_aucs))
print(f"\n95% CI Accuracy: [{ci_acc[0]:.4f}, {ci_acc[1]:.4f}]")
print(f"95% CI AUC:      [{ci_auc[0]:.4f}, {ci_auc[1]:.4f}]")


---
## Section 10: Ablation Study

Compare 4 configurations using **pre-computed intermediate features**:

| Config | Visual (768→256) | Pose (16D) | Adapter FT | Description |
|--------|:---:|:---:|:---:|-------------|
| A | ✅ | ✅ | ✅ | Full model (Partial FT baseline) |
| B | ✅ | zeroed | ✅ | VideoMAE only |
| C | zeroed | ✅ | ✅ | Pose only |
| D | ✅ (frozen proj) | ✅ | ❌ | Frozen VideoMAE (no domain adapter) |

All configs use the same LR scheduler for fair comparison.


In [ ]:
# ============================================================
# SECTION 10: Ablation Study
# ============================================================

def run_ablation_config(config_name, all_int_feats, pose_feats,
                        data_df, cfg, device,
                        zero_visual=False, zero_pose=False, use_causal=True,
                        freeze_adapter=False):
    """
    Run a single ablation configuration using pre-computed intermediate features.
    Features are zeroed out to test component contribution — model
    architecture stays identical (same input dim) for fair comparison.
    """
    print(f"\n{'='*60}")
    print(f"ABLATION: {config_name}")
    print(f"  Visual:  {'ZEROED' if zero_visual else 'ON'}")
    print(f"  Pose:    {'ZEROED' if zero_pose else 'ON'}")
    print(f"  Adapter: {'FROZEN' if freeze_adapter else 'TRAINABLE'}")
    print(f"  Causal:  {'ON' if use_causal else 'OFF'}")
    print(f"{'='*60}")

    import copy
    data_labels = data_df["label"].values
    animal_ids = data_df["animal_id"].values
    video_paths = data_df["video_path"].values

    n_healthy = (data_labels == 0).sum()
    n_lame = (data_labels == 1).sum()
    pos_weight = torch.tensor([n_healthy / n_lame]).to(device)

    cv = StratifiedGroupKFold(n_splits=cfg["CV_FOLDS"], shuffle=True,
                               random_state=cfg["SEED"])

    fold_metrics = []
    for fold, (train_idx, val_idx) in enumerate(
            cv.split(video_paths, data_labels, animal_ids)):

        # Apply feature zeroing
        train_vis = [np.zeros_like(all_int_feats[i]) if zero_visual
                     else all_int_feats[i] for i in train_idx]
        val_vis = [np.zeros_like(all_int_feats[i]) if zero_visual
                   else all_int_feats[i] for i in val_idx]

        train_pose = (np.zeros_like(pose_feats[train_idx]) if zero_pose
                      else pose_feats[train_idx])
        val_pose = (np.zeros_like(pose_feats[val_idx]) if zero_pose
                    else pose_feats[val_idx])

        train_ds = CowLamenessDatasetV32(
            train_vis, train_pose, data_labels[train_idx], cfg)
        val_ds = CowLamenessDatasetV32(
            val_vis, val_pose, data_labels[val_idx], cfg)

        train_loader = DataLoader(train_ds, batch_size=cfg["BATCH_SIZE"],
                                   shuffle=True, collate_fn=collate_fn, num_workers=0)
        val_loader = DataLoader(val_ds, batch_size=cfg["BATCH_SIZE"],
                                 shuffle=False, collate_fn=collate_fn, num_workers=0)

        # Fresh adapter + model per fold
        # Fresh adapter per fold via deepcopy
        abl_adapter = copy.deepcopy(domain_adapter)

        if freeze_adapter:
            for p in abl_adapter.parameters():
                p.requires_grad = False

        abl_model = CowLamenessModelV32(
            adapter=abl_adapter,
            pose_dim=cfg["POSE_FEAT_DIM"],
            hidden_dim=cfg["HIDDEN_DIM"],
            num_heads=cfg["NUM_HEADS"],
            num_layers=cfg["NUM_LAYERS"],
            dropout=cfg["DROPOUT"],
        ).to(device)

        # Optimizer with scheduler
        trainable_params = [p for p in abl_model.parameters() if p.requires_grad]
        if not freeze_adapter:
            adapter_p = [p for p in abl_model.adapter.parameters() if p.requires_grad]
            temporal_p = [p for n, p in abl_model.named_parameters()
                         if not n.startswith("adapter.") and p.requires_grad]
            optimizer = torch.optim.AdamW([
                {"params": adapter_p, "lr": cfg["LR_VIDEOMAE"]},
                {"params": temporal_p, "lr": cfg["LR_HEAD"]},
            ], weight_decay=cfg["WEIGHT_DECAY"])
        else:
            optimizer = torch.optim.AdamW(
                trainable_params, lr=cfg["LR_HEAD"],
                weight_decay=cfg["WEIGHT_DECAY"])

        criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
            optimizer, mode='min', factor=0.5, patience=3
        )

        best_val_loss = float('inf')
        patience_counter = 0
        best_metrics = None

        for epoch in range(cfg["EPOCHS"]):
            abl_model.train()
            for vis_b, pose_b, lbl_b, mask_b in train_loader:
                vis_b = vis_b.to(device)
                pose_b = pose_b.to(device)
                lbl_b = lbl_b.float().to(device)
                mask_b = mask_b.to(device)

                optimizer.zero_grad()
                logits, _ = abl_model(vis_b, pose_b, padding_mask=mask_b,
                                      use_causal=use_causal)
                loss = criterion(logits.squeeze(-1), lbl_b)
                loss.backward()
                torch.nn.utils.clip_grad_norm_(abl_model.parameters(),
                                               cfg["GRAD_CLIP"])
                optimizer.step()

            val_m, _, _, _ = evaluate(abl_model, val_loader, criterion, device,
                                      use_causal=use_causal)

            scheduler.step(val_m["loss"])

            if val_m["loss"] < best_val_loss - 0.001:
                best_val_loss = val_m["loss"]
                patience_counter = 0
                best_metrics = val_m.copy()
            else:
                patience_counter += 1

            if patience_counter >= cfg["PATIENCE"]:
                break

        fold_metrics.append(best_metrics)
        print(f"  Fold {fold+1}: Acc={best_metrics['accuracy']:.3f} "
              f"F1={best_metrics['f1']:.3f} AUC={best_metrics['auc']:.3f}")

        # Free GPU memory after each ablation fold
        del abl_model, abl_adapter
        torch.cuda.empty_cache()

    # Average
    result = {"config": config_name}
    for key in ["accuracy", "precision", "recall", "f1", "auc"]:
        vals = [m[key] for m in fold_metrics]
        result[key] = f"{np.mean(vals):.4f}±{np.std(vals):.4f}"

    return result


# Run ablation study
print("\n🔬 Starting Ablation Study (4 configs × 5 folds)")
print("Using pre-computed intermediate features + domain adapter.\n")

ablation_results = []

# Config A: Full model (use results from main training)
ablation_results.append({
    "config": "A: Full (Partial FT + Pose)",
    "accuracy": f"{means['accuracy']:.4f}±{stds['accuracy']:.4f}",
    "precision": f"{means['precision']:.4f}±{stds['precision']:.4f}",
    "recall": f"{means['recall']:.4f}±{stds['recall']:.4f}",
    "f1": f"{means['f1']:.4f}±{stds['f1']:.4f}",
    "auc": f"{means['auc']:.4f}±{stds['auc']:.4f}",
})

# Config B: VideoMAE only (zero pose)
result_b = run_ablation_config(
    "B: VideoMAE Only", all_intermediate_features, pose_features,
    data_df, CFG, DEVICE, zero_pose=True)
ablation_results.append(result_b)

# Config C: Pose only (zero visual)
result_c = run_ablation_config(
    "C: Pose Only", all_intermediate_features, pose_features,
    data_df, CFG, DEVICE, zero_visual=True)
ablation_results.append(result_c)

# Config D: Frozen VideoMAE (no domain adapter training)
result_d = run_ablation_config(
    "D: Frozen VideoMAE", all_intermediate_features, pose_features,
    data_df, CFG, DEVICE, freeze_adapter=True)
ablation_results.append(result_d)


In [ ]:
# ============================================================
# Ablation Results Table
# ============================================================

abl_df = pd.DataFrame(ablation_results)
print("\n" + "="*80)
print("📊 ABLATION STUDY RESULTS")
print("="*80)
print(abl_df.to_string(index=False))
print("="*80)

abl_df.to_csv(os.path.join(CFG["RESULTS_DIR"], "ablation_results.csv"), index=False)
print(f"💾 Saved ablation results")


---
## Section 11: Explainability & Feature Analysis

- **Temporal attention bar chart** — which clips are most important (3 correct + 3 incorrect)
- **Pose feature importance** via label correlation
- **Feature statistical comparison** with Welch's t-test and Cohen's d


In [ ]:
# ============================================================
# SECTION 11: Temporal Attention Visualization
# ============================================================

def plot_temporal_attention(fold_results, save_path=None):
    """
    Temporal attention bar charts for 3 correct + 3 incorrect predictions.
    Shows which clips the model focuses on for each video.
    """
    # Use last fold results (most complete)
    last_fold = fold_results[-1]
    if "fold_attns" not in last_fold:
        print("⚠️ No attention data available for visualization")
        return

    attns = last_fold["fold_attns"]   # (N_val, num_clips)
    probs = last_fold["fold_probs"]
    labels = last_fold["fold_labels"]

    preds = (probs >= 0.5).astype(int)
    correct_mask = preds == labels
    incorrect_mask = ~correct_mask

    # Select up to 3 correct and 3 incorrect
    correct_idx = np.where(correct_mask)[0][:3]
    incorrect_idx = np.where(incorrect_mask)[0][:3]

    n_correct = len(correct_idx)
    n_incorrect = len(incorrect_idx)
    n_total = n_correct + n_incorrect

    if n_total == 0:
        print("⚠️ No predictions to visualize")
        return

    fig, axes = plt.subplots(2, max(n_correct, n_incorrect, 1),
                              figsize=(6 * max(n_correct, n_incorrect, 1), 8))
    if max(n_correct, n_incorrect, 1) == 1:
        axes = axes.reshape(2, 1)

    # Correct predictions
    for i in range(max(n_correct, n_incorrect, 1)):
        # Top row: correct
        ax = axes[0, i]
        if i < n_correct:
            idx = correct_idx[i]
            attn_vals = attns[idx]
            true_label = "Lame" if labels[idx] == 1 else "Healthy"
            prob_val = probs[idx]
            colors = ['#2ecc71'] * len(attn_vals)
            ax.bar(range(len(attn_vals)), attn_vals, color=colors, alpha=0.8)
            ax.set_title(f"✅ Correct: {true_label}\np={prob_val:.3f}",
                        fontsize=10, fontweight='bold')
            ax.set_xlabel("Clip Index", fontsize=9)
            ax.set_ylabel("Attention Weight", fontsize=9)
            ax.set_ylim(0, max(attn_vals) * 1.2)
        else:
            ax.set_visible(False)

        # Bottom row: incorrect
        ax = axes[1, i]
        if i < n_incorrect:
            idx = incorrect_idx[i]
            attn_vals = attns[idx]
            true_label = "Lame" if labels[idx] == 1 else "Healthy"
            pred_label = "Lame" if preds[idx] == 1 else "Healthy"
            prob_val = probs[idx]
            colors = ['#e74c3c'] * len(attn_vals)
            ax.bar(range(len(attn_vals)), attn_vals, color=colors, alpha=0.8)
            ax.set_title(f"❌ Wrong: True={true_label}, Pred={pred_label}\n"
                        f"p={prob_val:.3f}", fontsize=10, fontweight='bold')
            ax.set_xlabel("Clip Index", fontsize=9)
            ax.set_ylabel("Attention Weight", fontsize=9)
            ax.set_ylim(0, max(attn_vals) * 1.2)
        else:
            ax.set_visible(False)

    plt.suptitle("Temporal Attention Weights (Which Clips Matter?)",
                 fontsize=14, fontweight='bold')
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.show()

plot_temporal_attention(fold_results,
    save_path=os.path.join(CFG["RESULTS_DIR"], "temporal_attention.png"))


In [ ]:
# ============================================================
# Pose Feature Statistical Analysis
# ============================================================

def pose_feature_ttest_table(features, data_labels, feature_names, save_path=None):
    """Welch's t-test for each pose feature: healthy vs lame."""
    rows = []
    for i, name in enumerate(feature_names):
        h = features[data_labels == 0, i]
        l = features[data_labels == 1, i]
        h = h[~np.isnan(h)]  # Exclude NaN (missing data)
        l = l[~np.isnan(l)]

        if len(h) > 5 and len(l) > 5:
            t_stat, p_val = stats.ttest_ind(h, l, equal_var=False)
            pooled_std = np.sqrt((np.var(h) + np.var(l)) / 2 + 1e-8)
            effect_size = (np.mean(l) - np.mean(h)) / pooled_std
            rows.append({
                "Feature": name,
                "Healthy (mean±std)": f"{np.mean(h):.4f}±{np.std(h):.4f}",
                "Lame (mean±std)": f"{np.mean(l):.4f}±{np.std(l):.4f}",
                "t-stat": f"{t_stat:.3f}",
                "p-value": f"{p_val:.6f}",
                "Cohen's d": f"{effect_size:.3f}",
                "Sig.": "***" if p_val < 0.001 else "**" if p_val < 0.01 else "*" if p_val < 0.05 else "ns"
            })

    df = pd.DataFrame(rows)
    print("\n📊 Pose Feature Statistical Comparison (Welch's t-test)")
    print("="*100)
    print(df.to_string(index=False))
    print("="*100)

    if save_path:
        df.to_csv(save_path, index=False)
    return df

ttest_df = pose_feature_ttest_table(
    pose_features, data_df["label"].values, pose_extractor.FEATURE_NAMES,
    save_path=os.path.join(CFG["RESULTS_DIR"], "pose_feature_ttest.csv")
)


In [ ]:
# ============================================================
# Feature Importance (correlation with label)
# ============================================================

def plot_feature_importance(features, data_labels, feature_names, save_path=None):
    """
    Feature importance: absolute correlation between each pose feature
    and the binary label.
    """
    fig, ax = plt.subplots(figsize=(12, 6))

    importance = []
    for i, name in enumerate(feature_names):
        feat_vals = features[:, i]
        valid_mask = ~np.isnan(feat_vals)
        if valid_mask.sum() > 5 and np.std(feat_vals[valid_mask]) > 0:
            corr = abs(np.corrcoef(feat_vals[valid_mask], data_labels[valid_mask])[0, 1])
        else:
            corr = 0.0
        importance.append(corr)

    sorted_idx = np.argsort(importance)[::-1]
    sorted_names = [feature_names[i] for i in sorted_idx]
    sorted_vals = [importance[i] for i in sorted_idx]

    colors = ['#e74c3c' if v > 0.1 else '#3498db' if v > 0.05 else '#95a5a6'
              for v in sorted_vals]
    ax.barh(range(len(sorted_names)), sorted_vals, color=colors)
    ax.set_yticks(range(len(sorted_names)))
    ax.set_yticklabels(sorted_names, fontsize=10)
    ax.set_xlabel('|Correlation with Label|', fontsize=12)
    ax.set_title('Pose Feature Importance (Correlation with Lameness)',
                 fontsize=13, fontweight='bold')
    ax.invert_yaxis()
    ax.grid(True, axis='x', alpha=0.3)

    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.show()

plot_feature_importance(
    pose_features, data_df["label"].values, pose_extractor.FEATURE_NAMES,
    save_path=os.path.join(CFG["RESULTS_DIR"], "feature_importance.png")
)


---
## Section 12: Results Summary & Model Export


In [ ]:
# ============================================================
# SECTION 12: Final Results Summary
# ============================================================

print("\n" + "="*80)
print("🏆 FINAL RESULTS SUMMARY — Cow Lameness Analysis v32")
print("="*80)

print(f"\n📐 Architecture:")
print(f"   VideoMAE (blocks 0-9 frozen, 10-11 trainable) → {CFG['PROJECTION_DIM']}D")
print(f"   + DLC Pose ({CFG['POSE_FEAT_DIM']}D)")
print(f"   → Temporal Transformer ({CFG['NUM_LAYERS']}L, {CFG['NUM_HEADS']}H)")
print(f"   → Binary Classification (Healthy/Lame)")

print(f"\n📊 Dataset:")
print(f"   Total videos: {len(data_df)}")
print(f"   Healthy: {(data_df['label']==0).sum()} | Lame: {(data_df['label']==1).sum()}")
print(f"   Unique animals: {data_df['animal_id'].nunique()}")
print(f"   Validation: {CFG['CV_FOLDS']}-fold subject-level CV")

print(f"\n🎯 Performance (Mean ± Std across {CFG['CV_FOLDS']} folds):")
print(f"   Accuracy:  {means['accuracy']:.4f} ± {stds['accuracy']:.4f}")
print(f"   Precision: {means['precision']:.4f} ± {stds['precision']:.4f}")
print(f"   Recall:    {means['recall']:.4f} ± {stds['recall']:.4f}")
print(f"   F1 Score:  {means['f1']:.4f} ± {stds['f1']:.4f}")
print(f"   AUC-ROC:   {means['auc']:.4f} ± {stds['auc']:.4f}")

target_met = means['accuracy'] >= 0.80
print(f"\n{'✅' if target_met else '❌'} Target accuracy ≥ 80%: {'MET' if target_met else 'NOT MET'}")

print(f"\n📁 All results saved to: {CFG['RESULTS_DIR']}")
print("="*80)


In [ ]:
# ============================================================
# Save Best Model
# ============================================================

# Select best fold by F1
best_fold_idx = np.argmax([r["f1"] for r in fold_results])
best_model_state = best_models[best_fold_idx]

save_dict = {
    "cfg": CFG,
    "model_state": best_model_state["model"],
    "fold_results": [{k: v for k, v in r.items()
                      if k not in ("history", "fold_probs", "fold_labels", "fold_attns")}
                     for r in fold_results],
    "means": means,
    "stds": stds,
    "pose_feature_names": pose_extractor.FEATURE_NAMES,
}

model_path = os.path.join(CFG["RESULTS_DIR"], "best_model_v32.pth")
torch.save(save_dict, model_path)
print(f"💾 Best model saved: {model_path}")
print(f"   Best fold: {best_fold_idx + 1} (F1={fold_results[best_fold_idx]['f1']:.4f})")

# Save all results as JSON
import json as json_lib
results_summary = {
    "version": "v32",
    "architecture": "PartialFT_VideoMAE(blocks10-11) + DLC_Pose + TemporalTransformer",
    "classification": "binary",
    "dataset_size": len(data_df),
    "cv_folds": CFG["CV_FOLDS"],
    "means": {k: float(v) for k, v in means.items()},
    "stds": {k: float(v) for k, v in stds.items()},
}
with open(os.path.join(CFG["RESULTS_DIR"], "results_summary.json"), "w") as f:
    json_lib.dump(results_summary, f, indent=2)

print("\n✅ All artifacts saved. Notebook execution complete.")
